In [1]:
import os
import GEOparse
import pandas as pd


# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {"sample_title": sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':')
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


# Create metadata DataFrame and join as columns to df_genes (rows=samples)
metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'

# Apply the cohort-selection rules from the paper:
# 1) keep only secondary high-grade glioma samples (grade == HGG)
# 2) remove low-quality samples with uniquely mapped reads < 30%
# 3) if a patient has multiple runs, keep the run with best mapping quality
col_map = 'percentage of uniquely mapped reads'

metadata_df = (
    metadata_df
    .assign(
        # Normalize fields used in filtering/sorting to avoid type/whitespace issues.
        grade=lambda d: d['grade'].astype(str).str.strip(),
        **{col_map: lambda d: pd.to_numeric(d[col_map])},
        **{'patient id': lambda d: d['patient id'].astype(str).str.strip()},
    )
    # Apply diagnosis + QC criteria in one step.
    .query("grade == 'HGG' and `percentage of uniquely mapped reads` >= 30")
    # Sort so best-quality sample (highest mapping rate) is first per patient.
    .sort_values(['patient id', col_map], ascending=[True, False])
    # Keep exactly one sample per patient.
    .drop_duplicates('patient id', keep='first')
    .copy()
)


In [2]:
# Sanity checks for cohort selection
print(f"Final selected rows: {metadata_df.shape[0]}")
print(f"Unique patients: {metadata_df['patient id'].nunique()}")
print(f"Duplicate patient IDs remaining: {metadata_df['patient id'].duplicated().sum()}")
print(
    "Min unique mapping (%): "
    f"{metadata_df['percentage of uniquely mapped reads'].min():.2f}"
)


Final selected rows: 60
Unique patients: 60
Duplicate patient IDs remaining: 0
Min unique mapping (%): 31.59


# Use NCBI provided expression data

In [3]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"

df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)

data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined = metadata_df.join(df_raw_counts, how='inner')


df_combined.reset_index()
df_combined.to_csv(data_file_name, index=False)

df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_1569/1012403528.py:10: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,tissue,unique sample id,patient id,grade,batch,idh status prediction,age,date of birth,vital status,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM5600930,Sample_S-10-33895_S22_L003merged,Glioma,Sample_S-10-33895,1,HGG,1,Mutant,22,3/9/1979,1,...,20285,98,157,122,14226,3986,791,9529,34,614
GSM5600825,Sample_R378-14-133900_S12_L002,Glioma,Sample_14-133900,10,HGG,2,WT,39,2/24/1970,1,...,14107,32,51,43,8383,1300,309,4762,61,1948
GSM5600933,Sample_S-12-53702_S22_L003merged,Glioma,Sample_S-12-53702,103,HGG,1,WT,59,8/12/1950,1,...,50527,227,329,258,40088,6765,1448,20185,153,2319
GSM5600934,Sample_S-13-16523_S22_L003merged,Glioma,Sample_S-13-16523,104,HGG,1,Mutant,31,2/11/1966,1,...,68629,192,291,288,50834,10367,1439,38179,123,1110
GSM5600826,Sample_R378-15-49530_S13_L002,Glioma,Sample_15-49530,11,HGG,2,Mutant,49,1/21/1956,0,...,7757,52,117,113,5845,1466,247,4186,47,807
GSM5600769,Sample_16-647051_S49_L005,Glioma,Sample_16-647051,12,HGG,3,Mutant,20,1/21/1986,1,...,48912,114,157,100,34258,7266,2305,20248,138,6669
GSM5600770,Sample_17-276430_S51_L006,Glioma,Sample_17-276430,13,HGG,3,Mutant,51,5/3/1956,0,...,29135,181,203,119,14702,1083,372,18499,178,2771
GSM5600828,Sample_R378-17-329452_S15_L002,Glioma,Sample_17-329452,14,HGG,2,Mutant,32,8/23/1978,0,...,64896,106,94,54,26187,3565,463,37503,119,6730
GSM5600773,Sample_18-280601_S42_L005,Glioma,Sample_18-280601,15,HGG,2,Mutant,56,7/31/1954,1,...,85674,502,853,598,27707,3642,846,50929,177,16484


# Generate description

In [4]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_geo

description_file_path = 'description.json'


if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_geo(GEO_ID)

desc_data['data_summary'] = {
    'samples': metadata_df.shape[0],
    'clinical_features': metadata_df.shape[1],
    'genes': df_raw_counts.shape[1]
}

print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)

Description updated


# Use author provided expression data

In [5]:
metadata_df = metadata_df.reset_index()
metadata_df = metadata_df.set_index('sample_title')

raw_exp_url = 'https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE184941&format=file&file=GSE184941%5FRaw%5Fgene%5Fcounts%5Fmatrix%2Etxt%2Egz'

df_raw_exp: pd.DataFrame = pd.read_csv(raw_exp_url, sep='\t', index_col=0, compression='gzip')
df_raw_exp = df_raw_exp[df_raw_exp.index.notna()]
df_raw_exp = df_raw_exp.T
df_raw_exp.index.name = "samples"
df_raw_exp.columns.name = None

df_combined = metadata_df.join(df_raw_exp, how='inner')
df_combined.reset_index()
# INFO: Uncomment to save the data to disk
# df_combined.to_csv(f'{GEO_ID}_raw.csv', index=False)

df_combined

,samples,tissue,unique sample id,patient id,grade,batch,idh status prediction,age,date of birth,vital status,...,ENSG00000198886.2,ENSG00000210176.1,ENSG00000210184.1,ENSG00000210191.1,ENSG00000198786.2,ENSG00000198695.2,ENSG00000210194.1,ENSG00000198727.2,ENSG00000210195.2,ENSG00000210196.2
Sample_S-10-33895_S22_L003merged,GSM5600930,Glioma,Sample_S-10-33895,1,HGG,1,Mutant,22,3/9/1979,1,...,38142,2,0,0,26053,461,0,19026,17,0
Sample_R378-14-133900_S12_L002,GSM5600825,Glioma,Sample_14-133900,10,HGG,2,WT,39,2/24/1970,1,...,30030,422,0,0,17789,196,0,10039,129,0
Sample_S-12-53702_S22_L003merged,GSM5600933,Glioma,Sample_S-12-53702,103,HGG,1,WT,59,8/12/1950,1,...,94812,0,0,0,71512,1658,40,39213,89,60
Sample_S-13-16523_S22_L003merged,GSM5600934,Glioma,Sample_S-13-16523,104,HGG,1,Mutant,31,2/11/1966,1,...,127744,2,0,0,93925,3050,355,74606,51,85
Sample_R378-15-49530_S13_L002,GSM5600826,Glioma,Sample_15-49530,11,HGG,2,Mutant,49,1/21/1956,0,...,14689,469,0,0,9681,602,12,8639,57,42
Sample_16-647051_S49_L005,GSM5600769,Glioma,Sample_16-647051,12,HGG,3,Mutant,20,1/21/1986,1,...,105009,1350,3,19,73672,1565,60,45039,257,191
Sample_17-276430_S51_L006,GSM5600770,Glioma,Sample_17-276430,13,HGG,3,Mutant,51,5/3/1956,0,...,57455,3028,37,128,26788,351,23,38020,227,181
Sample_R378-17-329452_S15_L002,GSM5600828,Glioma,Sample_17-329452,14,HGG,2,Mutant,32,8/23/1978,0,...,132691,2607,6,6,52715,710,0,78905,52,0
Sample_18-280601_S42_L005,GSM5600773,Glioma,Sample_18-280601,15,HGG,2,Mutant,56,7/31/1954,1,...,160603,40,0,0,46847,946,6,102807,61,70
Sample_R378-19-473907_S16_L002,GSM5600829,Glioma,Sample_19-473907,16,HGG,2,WT,65,12/7/1940,1,...,17952,740,0,0,15720,1298,0,9414,4,28
